# Lib


In [ ]:
import os
import copy
import random
from pathlib import Path
from datetime import datetime

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import albumentations as A
from albumentations.pytorch import ToTensorV2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from torchvision.models import resnet50, ResNet50_Weights, resnet34, ResNet34_Weights
from scipy.ndimage import binary_erosion, distance_transform_edt

import kagglehub

In [ ]:
SEED = 42


def set_seed(seed=SEED):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except Exception:
        pass


def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


set_seed(SEED)
print(f"Global seed: {SEED}")

# Config


## Default


In [ ]:
DEFAULT_CONFIG = {
    "seed": SEED,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    # Input
    "image_height": 224,
    "image_width": 224,
    "num_classes": 3,
    # U-Net
    "backbone": "resnet50",
    "pretrained": True,
    "bilinear": False,
    "freeze_encoder": False,
    # Semi-supervised train split: "half" or "quarter"
    "train_subset": "half",
    # Separate loaders make the labeled, unlabeled ratio explicit
    "labeled_batch_size": 16,
    "unlabeled_batch_size": 16,
    "eval_batch_size": 16,
    "num_workers": 2,
    # Training
    "epochs": 50,
    "save_every": 35,
    # Early stopping uses Teacher validation loss
    "early_stopping": True,
    "early_stopping_patience": 10,
    "early_stopping_min_delta": 0.0,
    # CSV columns
    "image_col": None,
    "mask_col": None,
    "timestamp_format": "%y%m%d_%H%M%S",
}

## Mean teacher


In [ ]:
DEFAULT_MEAN_TEACHER_CONFIG = {
    "ema_decay": 0.99,
    "max_consistency_weight": 5,
    "consistency_rampup_epochs": 30,
}

## Path


In [ ]:
PROJECT_ROOT = Path("/kaggle/working")
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints_mean_teacher"
MODEL_DIR = PROJECT_ROOT / "models_mean_teacher"

# None = fresh training
RESUME_CHECKPOINT = None

# Data augmentation


In [ ]:
import cv2
import albumentations as A
from albumentations.pytorch import ToTensorV2

SEED = 42  # Đảm bảo bạn đã cấu hình SEED


def build_labeled_shared_geometry(height=224, width=224, seed=SEED):
    """Geometric-only view applied jointly to image and mask."""
    return A.Compose(
        [
            A.Resize(height=height, width=width),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.RandomRotate90(p=0.5),
            A.Affine(
                translate_percent={"x": (-0.0625, 0.0625), "y": (-0.0625, 0.0625)},
                scale=(0.85, 1.15),
                rotate=(-45, 45),
                border_mode=cv2.BORDER_CONSTANT,  # Sửa 'mode' thành 'border_mode' ở đây
                p=0.5,
            ),
            A.OneOf(
                [
                    A.ElasticTransform(
                        alpha=120,
                        sigma=120 * 0.05,
                        p=0.5,
                    ),
                    A.GridDistortion(p=0.5),
                    A.OpticalDistortion(
                        distort_limit=(-1.0, 1.0),
                        p=0.5,
                    ),
                ],
                p=0.4,
            ),
        ],
        seed=int(seed),
        strict=True,
    )


def build_labeled_weak_transform(seed=SEED):
    """Teacher view of labeled images: normalize only, no photometric noise."""
    return A.Compose(
        [
            A.Normalize(
                mean=(0.485, 0.456, 0.406),
                std=(0.229, 0.224, 0.225),
            ),
            ToTensorV2(),
        ],
        seed=int(seed),
        strict=True,
    )


def build_labeled_strong_transform(seed=SEED):
    """Student view: photometric-only perturbations after shared geometry."""
    return A.Compose(
        [
            A.OneOf(
                [
                    A.RandomBrightnessContrast(
                        brightness_limit=(-0.10, 0.10),
                        contrast_limit=(-0.10, 0.10),
                        p=0.5,
                    ),
                    A.HueSaturationValue(
                        hue_shift_limit=(-10, 10),
                        sat_shift_limit=(-15, 15),
                        val_shift_limit=(-10, 10),
                        p=0.5,
                    ),
                ],
                p=0.3,
            ),
            A.OneOf(
                [
                    A.GaussNoise(p=0.5),
                    A.GaussianBlur(blur_limit=(3, 5), p=0.5),
                ],
                p=0.2,
            ),
            A.Normalize(
                mean=(0.485, 0.456, 0.406),
                std=(0.229, 0.224, 0.225),
            ),
            ToTensorV2(),
        ],
        seed=int(seed),
        strict=True,
    )


def build_unlabeled_shared_geometry(height=224, width=224, seed=SEED):
    """One geometric view shared by Teacher and Student to preserve alignment."""
    return A.Compose(
        [
            A.Resize(height=height, width=width),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.RandomRotate90(p=0.5),
            A.Affine(
                translate_percent={"x": (-0.0625, 0.0625), "y": (-0.0625, 0.0625)},
                scale=(0.90, 1.10),
                rotate=(-20, 20),
                border_mode=cv2.BORDER_CONSTANT,  # Sửa 'mode' thành 'border_mode' ở đây
                p=0.5,
            ),
        ],
        seed=int(seed),
        strict=True,
    )


def build_unlabeled_weak_transform(seed=SEED):
    return A.Compose(
        [
            A.Normalize(
                mean=(0.485, 0.456, 0.406),
                std=(0.229, 0.224, 0.225),
            ),
            ToTensorV2(),
        ],
        seed=int(seed),
        strict=True,
    )


def build_unlabeled_strong_transform(seed=SEED):
    """Photometric-only perturbations keep Teacher/Student pixels aligned."""
    return A.Compose(
        [
            A.OneOf(
                [
                    A.RandomBrightnessContrast(
                        brightness_limit=(-0.20, 0.20),
                        contrast_limit=(-0.20, 0.20),
                        p=1.0,
                    ),
                    A.HueSaturationValue(
                        hue_shift_limit=(-15, 15),
                        sat_shift_limit=(-25, 25),
                        val_shift_limit=(-15, 15),
                        p=1.0,
                    ),
                ],
                p=0.8,
            ),
            A.OneOf(
                [
                    A.GaussNoise(p=1.0),
                    A.GaussianBlur(blur_limit=(3, 5), p=1.0),
                ],
                p=0.3,
            ),
            A.Normalize(
                mean=(0.485, 0.456, 0.406),
                std=(0.229, 0.224, 0.225),
            ),
            ToTensorV2(),
        ],
        seed=int(seed),
        strict=True,
    )


def build_eval_transform(height=224, width=224, seed=SEED):
    return A.Compose(
        [
            A.Resize(height=height, width=width),
            A.Normalize(
                mean=(0.485, 0.456, 0.406),
                std=(0.229, 0.224, 0.225),
            ),
            ToTensorV2(),
        ],
        seed=int(seed),
        strict=True,
    )

# Data preparation


In [ ]:
CLASS_NAMES = {
    0: "background",
    1: "non-neoplastic",
    2: "neoplastic",
}

CLASS_COLORS = {
    0: (0, 0, 0),
    1: (0, 255, 0),
    2: (255, 0, 0),
}


def decode_ground_truth(mask_rgb):
    """
    To RGB mask -> class-index mask 0/1/2.

    Class mapping:
        0 = background      -> black
        1 = non-neoplastic  -> green
        2 = neoplastic      -> red

    Using nearest-color matching to handle or correct any color variations in the mask image.
    This function will convert an RGB mask into a 2D array where each pixel's value corresponds to its class ID (0, 1, or 2).
    """
    mask = np.asarray(mask_rgb)

    if mask.ndim == 2:
        unique_values = np.unique(mask)

        if np.isin(unique_values, [0, 1, 2]).all():
            return mask.astype(np.uint8)

        raise ValueError(
            "Grayscale mask must contain only class IDs 0, 1, 2. "
            f"Found: {unique_values[:20]}"
        )

    if mask.ndim != 3 or mask.shape[-1] != 3:
        raise ValueError(f"Expected HxWx3 RGB mask, got shape={mask.shape}")

    colors = np.array(
        [
            [0, 0, 0],  # background
            [0, 255, 0],  # non-neoplastic
            [255, 0, 0],  # neoplastic
        ],
        dtype=np.float32,
    )

    pixels = mask.astype(np.float32)

    # Shape:
    # pixels -> [H, W, 1, 3]
    # colors -> [1, 1, 3, 3]
    # result -> [H, W, 3]
    diff = pixels[:, :, None, :] - colors[None, None, :, :]

    distances = np.sum(
        diff**2,
        axis=-1,
    )

    class_mask = np.argmin(
        distances,
        axis=-1,
    )

    return class_mask.astype(np.uint8)


def _pick_column(
    df,
    explicit_column=None,
    candidates=None,
    required=True,
):
    if explicit_column is not None:
        if explicit_column not in df.columns:
            raise KeyError(
                f"Column '{explicit_column}' not found. "
                f"Available: {df.columns.tolist()}"
            )

        return explicit_column

    candidates = candidates or []

    normalized = {str(col).strip().lower(): col for col in df.columns}

    for candidate in candidates:
        if candidate.lower() in normalized:
            return normalized[candidate.lower()]

    if len(df.columns) == 1:
        return df.columns[0]

    if required:
        raise KeyError(
            "Cannot infer CSV column automatically. "
            f"Available columns: {df.columns.tolist()}"
        )

    return None


class PolypSegmentationDataset(Dataset):
    IMAGE_COLUMN_CANDIDATES = [
        "image",
        "image_name",
        "image_path",
        "filename",
        "file_name",
        "file",
        "id",
    ]

    MASK_COLUMN_CANDIDATES = [
        "mask",
        "mask_name",
        "mask_path",
        "label",
        "label_path",
        "gt",
        "gt_path",
    ]

    def __init__(
        self,
        dataframe,
        image_dir,
        mask_dir=None,
        transform=None,
        image_col=None,
        mask_col=None,
        has_mask=True,
    ):
        self.df = dataframe.reset_index(drop=True).copy()

        self.image_dir = Path(image_dir)

        self.mask_dir = Path(mask_dir) if mask_dir is not None else None

        self.transform = transform
        self.has_mask = has_mask

        self.image_col = _pick_column(
            self.df,
            explicit_column=image_col,
            candidates=self.IMAGE_COLUMN_CANDIDATES,
            required=True,
        )

        if self.has_mask:
            if self.mask_dir is None:
                raise ValueError("mask_dir is required when has_mask=True.")

            self.mask_col = _pick_column(
                self.df,
                explicit_column=mask_col,
                candidates=self.MASK_COLUMN_CANDIDATES,
                required=True,
            )

            if self.mask_col == self.image_col:
                raise ValueError(
                    "Mask column resolved to the same column as the image column. "
                    "Provide an explicit mask_col or fix the CSV schema."
                )

        else:
            self.mask_col = None

    @staticmethod
    def _resolve_path(root_dir, value):
        candidate = Path(str(value))

        if candidate.is_absolute():
            return candidate

        return Path(root_dir) / str(value)

    def __len__(self):
        return len(self.df)

    def get_image_path(self, index):
        value = self.df.iloc[index][self.image_col]

        return self._resolve_path(
            self.image_dir,
            value,
        )

    def get_mask_path(self, index):
        if not self.has_mask:
            return None

        row = self.df.iloc[index]
        mask_value = row[self.mask_col]

        return self._resolve_path(
            self.mask_dir,
            mask_value,
        )

    def __getitem__(self, index):
        image_path = self.get_image_path(index)

        image_bgr = cv2.imread(
            str(image_path),
            cv2.IMREAD_COLOR,
        )

        if image_bgr is None:
            raise FileNotFoundError(f"Cannot read image: {image_path}")

        image = cv2.cvtColor(
            image_bgr,
            cv2.COLOR_BGR2RGB,
        )

        if not self.has_mask:
            if self.transform is not None:
                transformed = self.transform(image=image)

                image = transformed["image"]

            return {
                "image": image,
                "image_path": str(image_path),
                "image_name": image_path.name,
            }

        mask_path = self.get_mask_path(index)

        mask_raw = cv2.imread(
            str(mask_path),
            cv2.IMREAD_UNCHANGED,
        )

        if mask_raw is None:
            raise FileNotFoundError(f"Cannot read mask: {mask_path}")

        if mask_raw.ndim == 3:
            mask_raw = cv2.cvtColor(
                mask_raw,
                cv2.COLOR_BGR2RGB,
            )

        # Nearest-color RGB -> class ID.
        mask = decode_ground_truth(mask_raw)

        if self.transform is not None:
            transformed = self.transform(
                image=image,
                mask=mask,
            )

            image = transformed["image"]
            mask = transformed["mask"]

        if not torch.is_tensor(mask):
            mask = torch.from_numpy(mask)

        return {
            "image": image,
            "mask": mask.long(),
            "image_path": str(image_path),
            "mask_path": str(mask_path),
            "image_name": image_path.name,
        }

In [ ]:
class MeanTeacherUnlabeledDataset(Dataset):
    """Return aligned weak and strong views of the same unlabeled image."""

    IMAGE_COLUMN_CANDIDATES = PolypSegmentationDataset.IMAGE_COLUMN_CANDIDATES

    def __init__(
        self,
        dataframe,
        image_dir,
        shared_geometry,
        weak_transform,
        strong_transform,
        image_col=None,
    ):
        self.df = dataframe.reset_index(drop=True).copy()
        self.image_dir = Path(image_dir)
        self.shared_geometry = shared_geometry
        self.weak_transform = weak_transform
        self.strong_transform = strong_transform
        self.image_col = _pick_column(
            self.df,
            explicit_column=image_col,
            candidates=self.IMAGE_COLUMN_CANDIDATES,
            required=True,
        )

    def __len__(self):
        return len(self.df)

    def get_image_path(self, index):
        value = self.df.iloc[index][self.image_col]
        candidate = Path(str(value))
        return candidate if candidate.is_absolute() else self.image_dir / str(value)

    def __getitem__(self, index):
        image_path = self.get_image_path(index)
        image_bgr = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
        if image_bgr is None:
            raise FileNotFoundError(f"Cannot read image: {image_path}")

        image = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

        # Geometry is sampled exactly once. Only photometric appearance differs.
        aligned_image = self.shared_geometry(image=image)["image"]
        weak_image = self.weak_transform(image=aligned_image)["image"]
        strong_image = self.strong_transform(image=aligned_image)["image"]

        return {
            "weak_image": weak_image,
            "strong_image": strong_image,
            "image_path": str(image_path),
            "image_name": image_path.name,
        }


class MeanTeacherLabeledDataset(Dataset):
    """Return aligned weak (Teacher) and strong (Student) views of the same
    labeled image, plus the mask aligned to the shared geometry.

    This mirrors MeanTeacherUnlabeledDataset so labeled examples also
    receive a consistency signal, matching the original Mean Teacher design
    where the consistency cost is evaluated on all input data, not only the
    unlabeled subset.
    """

    IMAGE_COLUMN_CANDIDATES = PolypSegmentationDataset.IMAGE_COLUMN_CANDIDATES
    MASK_COLUMN_CANDIDATES = PolypSegmentationDataset.MASK_COLUMN_CANDIDATES

    def __init__(
        self,
        dataframe,
        image_dir,
        mask_dir,
        shared_geometry,
        weak_transform,
        strong_transform,
        image_col=None,
        mask_col=None,
    ):
        self.df = dataframe.reset_index(drop=True).copy()
        self.image_dir = Path(image_dir)
        self.mask_dir = Path(mask_dir)
        self.shared_geometry = shared_geometry
        self.weak_transform = weak_transform
        self.strong_transform = strong_transform
        self.image_col = _pick_column(
            self.df,
            explicit_column=image_col,
            candidates=self.IMAGE_COLUMN_CANDIDATES,
            required=True,
        )
        self.mask_col = _pick_column(
            self.df,
            explicit_column=mask_col,
            candidates=self.MASK_COLUMN_CANDIDATES,
            required=True,
        )

        if self.mask_col == self.image_col:
            raise ValueError(
                "Mask column resolved to the same column as the image column. "
                "Provide an explicit mask_col or fix the CSV schema."
            )

    def __len__(self):
        return len(self.df)

    def get_image_path(self, index):
        value = self.df.iloc[index][self.image_col]
        candidate = Path(str(value))
        return candidate if candidate.is_absolute() else self.image_dir / str(value)

    def get_mask_path(self, index):
        row = self.df.iloc[index]
        mask_value = row[self.mask_col]
        candidate = Path(str(mask_value))
        return candidate if candidate.is_absolute() else self.mask_dir / str(mask_value)

    def __getitem__(self, index):
        image_path = self.get_image_path(index)
        image_bgr = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
        if image_bgr is None:
            raise FileNotFoundError(f"Cannot read image: {image_path}")
        image = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

        mask_path = self.get_mask_path(index)
        mask_raw = cv2.imread(str(mask_path), cv2.IMREAD_UNCHANGED)
        if mask_raw is None:
            raise FileNotFoundError(f"Cannot read mask: {mask_path}")
        if mask_raw.ndim == 3:
            mask_raw = cv2.cvtColor(mask_raw, cv2.COLOR_BGR2RGB)
        mask = decode_ground_truth(mask_raw)

        # Geometry sampled once so mask, weak view and strong view stay aligned.
        aligned = self.shared_geometry(image=image, mask=mask)
        aligned_image, aligned_mask = aligned["image"], aligned["mask"]

        weak_image = self.weak_transform(image=aligned_image)["image"]
        strong_image = self.strong_transform(image=aligned_image)["image"]

        if not torch.is_tensor(aligned_mask):
            aligned_mask = torch.from_numpy(aligned_mask)

        return {
            "weak_image": weak_image,
            "strong_image": strong_image,
            "mask": aligned_mask.long(),
            "image_path": str(image_path),
            "mask_path": str(mask_path),
            "image_name": image_path.name,
        }


def validate_labeled_dataset(dataset, dataset_name="dataset", max_samples=8):
    """Fail early if image/mask columns or representative mask paths are wrong."""
    if getattr(dataset, "mask_col", None) is None:
        raise ValueError(f"[{dataset_name}] mask_col was not resolved.")

    if dataset.image_col == dataset.mask_col:
        raise ValueError(
            f"[{dataset_name}] image_col and mask_col are both '{dataset.image_col}'."
        )

    if len(dataset) == 0:
        raise ValueError(f"[{dataset_name}] dataset is empty.")

    sample_count = min(int(max_samples), len(dataset))
    indices = np.unique(np.linspace(0, len(dataset) - 1, num=sample_count, dtype=int))
    seen_classes = set()

    for index in indices:
        image_path = Path(dataset.get_image_path(int(index)))
        mask_path = Path(dataset.get_mask_path(int(index)))

        if not image_path.exists():
            raise FileNotFoundError(f"[{dataset_name}] Missing image: {image_path}")
        if not mask_path.exists():
            raise FileNotFoundError(f"[{dataset_name}] Missing mask: {mask_path}")

        if image_path.resolve() == mask_path.resolve():
            raise ValueError(
                f"[{dataset_name}] Image and mask resolve to the same file: {image_path}"
            )

        mask_raw = cv2.imread(str(mask_path), cv2.IMREAD_UNCHANGED)
        if mask_raw is None:
            raise FileNotFoundError(f"[{dataset_name}] Cannot read mask: {mask_path}")
        if mask_raw.ndim == 3:
            mask_raw = cv2.cvtColor(mask_raw, cv2.COLOR_BGR2RGB)

        decoded = decode_ground_truth(mask_raw)
        unique_classes = set(np.unique(decoded).tolist())
        if not unique_classes.issubset({0, 1, 2}):
            raise ValueError(
                f"[{dataset_name}] Invalid decoded class IDs {sorted(unique_classes)} "
                f"in mask {mask_path}"
            )
        seen_classes.update(unique_classes)

    print(
        f"[Mask check] {dataset_name}: image_col='{dataset.image_col}', "
        f"mask_col='{dataset.mask_col}', checked={len(indices)}, "
        f"classes_seen={sorted(seen_classes)}"
    )

# Architecture


## U-Net


In [ ]:
class ResNetEncoder(nn.Module):
    """
    ResNet encoder backbone.

    Supports:
        resnet34, resnet50

    Returns:
        [x0, x1, x2, x3, x4]
    """

    def __init__(
        self,
        backbone="resnet34",
        pretrained=True,
    ):
        super().__init__()

        backbone = backbone.lower()

        if backbone == "resnet34":
            weights = ResNet34_Weights.DEFAULT if pretrained else None
            model = resnet34(weights=weights)
            self.out_channels = [64, 64, 128, 256, 512]

        elif backbone == "resnet50":
            weights = ResNet50_Weights.DEFAULT if pretrained else None
            model = resnet50(weights=weights)
            self.out_channels = [64, 256, 512, 1024, 2048]

        else:
            raise ValueError(
                f"Unsupported backbone: {backbone}. "
                "Supported: resnet18, resnet34, "
                "resnet50, resnet101"
            )

        self.conv1 = model.conv1
        self.bn1 = model.bn1
        self.relu = model.relu
        self.maxpool = model.maxpool

        self.layer1 = model.layer1
        self.layer2 = model.layer2
        self.layer3 = model.layer3
        self.layer4 = model.layer4

    def forward(self, x):
        x0 = self.relu(self.bn1(self.conv1(x)))  # H/2

        x = self.maxpool(x0)  # H/4
        x1 = self.layer1(x)  # H/4
        x2 = self.layer2(x1)  # H/8
        x3 = self.layer3(x2)  # H/16
        x4 = self.layer4(x3)  # H/32

        return [x0, x1, x2, x3, x4]

    def freeze(self):
        for parameter in self.parameters():
            parameter.requires_grad = False

    def unfreeze(self):
        for parameter in self.parameters():
            parameter.requires_grad = True

In [ ]:
class DoubleConv(nn.Module):
    """
    (Conv2d -> BatchNorm -> ReLU) * 2
    """

    def __init__(
        self,
        in_channels,
        out_channels,
    ):
        super().__init__()

        self.double_conv = nn.Sequential(
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=3,
                padding=1,
                bias=False,
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(
                out_channels,
                out_channels,
                kernel_size=3,
                padding=1,
                bias=False,
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.double_conv(x)


class Up(nn.Module):
    """
    Upsample decoder feature, concat encoder skip, before DoubleConv.
    """

    def __init__(
        self,
        decoder_channels,
        skip_channels,
        out_channels,
        bilinear=False,
    ):
        super().__init__()

        if bilinear:
            self.up = nn.Sequential(
                nn.Upsample(
                    scale_factor=2,
                    mode="bilinear",
                    align_corners=True,
                ),
                nn.Conv2d(
                    decoder_channels,
                    out_channels,
                    kernel_size=1,
                ),
            )
        else:
            self.up = nn.ConvTranspose2d(
                decoder_channels,
                out_channels,
                kernel_size=2,
                stride=2,
            )

        self.conv = DoubleConv(
            out_channels + skip_channels,
            out_channels,
        )

    def forward(self, x, skip):
        x = self.up(x)

        diff_y = skip.size(2) - x.size(2)
        diff_x = skip.size(3) - x.size(3)

        if diff_x != 0 or diff_y != 0:
            x = F.pad(
                x,
                [
                    diff_x // 2,
                    diff_x - diff_x // 2,
                    diff_y // 2,
                    diff_y - diff_y // 2,
                ],
            )

        x = torch.cat([skip, x], dim=1)

        return self.conv(x)


class OutConv(nn.Module):
    def __init__(
        self,
        in_channels,
        n_classes,
    ):
        super().__init__()

        self.conv = nn.Conv2d(
            in_channels,
            n_classes,
            kernel_size=1,
        )

    def forward(self, x):
        return self.conv(x)


class UNet(nn.Module):
    """
    U-Net with ResNet encoder backbone.

    Output:
        logits: (B, n_classes, H, W)

    return_features=True:
        return encoder/bottleneck/decoder features.
    """

    def __init__(
        self,
        backbone="resnet34",
        n_classes=3,
        pretrained=True,
        bilinear=False,
    ):
        super().__init__()

        self.backbone_name = backbone
        self.n_classes = n_classes
        self.pretrained = pretrained
        self.bilinear = bilinear

        self.encoder = ResNetEncoder(
            backbone=backbone,
            pretrained=pretrained,
        )

        c0, c1, c2, c3, c4 = self.encoder.out_channels

        self.up4 = Up(
            decoder_channels=c4,
            skip_channels=c3,
            out_channels=256,
            bilinear=bilinear,
        )

        self.up3 = Up(
            decoder_channels=256,
            skip_channels=c2,
            out_channels=128,
            bilinear=bilinear,
        )

        self.up2 = Up(
            decoder_channels=128,
            skip_channels=c1,
            out_channels=64,
            bilinear=bilinear,
        )

        self.up1 = Up(
            decoder_channels=64,
            skip_channels=c0,
            out_channels=64,
            bilinear=bilinear,
        )

        if bilinear:
            self.final_up = nn.Sequential(
                nn.Upsample(
                    scale_factor=2,
                    mode="bilinear",
                    align_corners=True,
                ),
                nn.Conv2d(
                    64,
                    32,
                    kernel_size=1,
                ),
                nn.BatchNorm2d(32),
                nn.ReLU(inplace=True),
            )
        else:
            self.final_up = nn.Sequential(
                nn.ConvTranspose2d(
                    64,
                    32,
                    kernel_size=2,
                    stride=2,
                ),
                nn.BatchNorm2d(32),
                nn.ReLU(inplace=True),
            )

        self.outc = OutConv(
            32,
            n_classes,
        )

    def forward(
        self,
        x,
        return_features=False,
    ):
        input_size = x.shape[-2:]

        features = self.encoder(x)
        x0, x1, x2, x3, x4 = features

        x = self.up4(x4, x3)
        x = self.up3(x, x2)
        x = self.up2(x, x1)
        x = self.up1(x, x0)

        decoder_feature = self.final_up(x)

        logits = self.outc(decoder_feature)

        if logits.shape[-2:] != input_size:
            logits = F.interpolate(
                logits,
                size=input_size,
                mode="bilinear",
                align_corners=False,
            )

        if return_features:
            return logits, {
                "encoder_0": x0,
                "encoder_1": x1,
                "encoder_2": x2,
                "encoder_3": x3,
                "bottleneck": x4,
                "decoder": decoder_feature,
            }

        return logits

    def freeze_encoder(self):
        self.encoder.freeze()

    def unfreeze_encoder(self):
        self.encoder.unfreeze()

# Metrics and losses


## metrics


In [ ]:
def _confusion_counts_from_logits(
    logits,
    targets,
    num_classes=3,
):
    """
    Aggregate TP/FP/FN over all pixels in the supplied batch.
    Counts are returned per class and can be accumulated across batches.
    """
    preds = torch.argmax(logits, dim=1)

    tp = np.zeros(num_classes, dtype=np.float64)
    fp = np.zeros(num_classes, dtype=np.float64)
    fn = np.zeros(num_classes, dtype=np.float64)

    for class_id in range(num_classes):
        pred_c = preds == class_id
        true_c = targets == class_id

        tp[class_id] = (pred_c & true_c).sum().item()
        fp[class_id] = (pred_c & ~true_c).sum().item()
        fn[class_id] = (~pred_c & true_c).sum().item()

    return tp, fp, fn


def _macro_dice_from_counts(
    tp,
    fp,
    fn,
    include_background=False,
):
    """
    Macro Dice from dataset/epoch-level TP/FP/FN counts.

    A class whose denominator is zero over the whole evaluated set is skipped
    from the macro average, matching the existing notebook behavior for an
    entirely absent class without letting batch composition bias the score.
    """
    start_class = 0 if include_background else 1
    scores = []

    for class_id in range(start_class, len(tp)):
        denominator = 2.0 * tp[class_id] + fp[class_id] + fn[class_id]
        if denominator > 0:
            scores.append((2.0 * tp[class_id]) / denominator)

    return float(np.mean(scores)) if scores else 0.0


def multiclass_dice_score(
    logits,
    targets,
    num_classes=3,
    include_background=False,
):
    """
    Macro Dice computed from TP/FP/FN aggregated over the supplied batch.

    Training/validation functions accumulate these counts across the full
    epoch before reporting Dice, so the final score is independent of how the
    dataset happens to be split into batches.
    """
    tp, fp, fn = _confusion_counts_from_logits(
        logits,
        targets,
        num_classes=num_classes,
    )
    score = _macro_dice_from_counts(
        tp,
        fp,
        fn,
        include_background=include_background,
    )
    return torch.tensor(score, dtype=torch.float32, device=logits.device)


def hd95_binary(pred_mask, true_mask):
    """
    Symmetric 95th-percentile Hausdorff distance for one binary mask pair.

    Empty-mask convention used in this notebook:
        - both GT and prediction empty -> 0.0
        - exactly one side empty       -> full image-diagonal penalty
        - both non-empty               -> standard symmetric HD95

    HD95 is computed per image/class and averaged later by evaluate_model().
    """
    pred = np.asarray(pred_mask).astype(bool)
    true = np.asarray(true_mask).astype(bool)

    pred_empty = not pred.any()
    true_empty = not true.any()

    if pred_empty and true_empty:
        return 0.0

    if pred_empty or true_empty:
        height, width = true.shape[-2:]
        return float(np.hypot(height, width))

    pred_surface = pred ^ binary_erosion(pred)
    true_surface = true ^ binary_erosion(true)

    if not pred_surface.any():
        pred_surface = pred
    if not true_surface.any():
        true_surface = true

    dist_to_true = distance_transform_edt(~true_surface)
    dist_to_pred = distance_transform_edt(~pred_surface)

    distances_pred_to_true = dist_to_true[pred_surface]
    distances_true_to_pred = dist_to_pred[true_surface]

    all_surface_distances = np.concatenate(
        [distances_pred_to_true, distances_true_to_pred]
    )

    return float(np.percentile(all_surface_distances, 95))


def _safe_divide(numerator, denominator):
    if denominator == 0:
        return np.nan
    return float(numerator / denominator)


## losses


In [ ]:
class GeneralizedDiceLoss(nn.Module):
    """Generalized Dice Loss (GDL): Automatically balances class weights based on the inverse square of the true area."""

    def __init__(self, num_classes=3, smooth=1e-6, ignore_index=None):
        super().__init__()
        self.num_classes = num_classes
        self.smooth = smooth
        self.ignore_index = ignore_index

    def forward(self, logits, targets):
        probabilities = torch.softmax(logits, dim=1)

        if self.ignore_index is not None:
            valid_mask = targets != self.ignore_index
            safe_targets = torch.where(valid_mask, targets, torch.zeros_like(targets))
        else:
            valid_mask = torch.ones_like(targets, dtype=torch.bool)
            safe_targets = targets

        targets_one_hot = F.one_hot(safe_targets.long(), num_classes=self.num_classes)
        targets_one_hot = targets_one_hot.permute(0, 3, 1, 2).float()
        valid_mask = valid_mask.unsqueeze(1).float()
        probabilities = probabilities * valid_mask
        targets_one_hot = targets_one_hot * valid_mask

        dims = (0, 2, 3)
        targets_sum = torch.sum(targets_one_hot, dim=dims)

        present = targets_sum > 0
        safe_targets_sum = torch.where(
            present,
            targets_sum,
            torch.ones_like(targets_sum),
        )
        weights = present.to(probabilities.dtype) / (
            safe_targets_sum + self.smooth
        ).pow(2)
        weights = weights / weights.sum().clamp_min(self.smooth)

        intersection = torch.sum(probabilities * targets_one_hot, dim=dims)
        pred_sum = torch.sum(probabilities, dim=dims)

        weighted_intersection = torch.sum(weights * intersection)
        weighted_denominator = torch.sum(weights * (pred_sum + targets_sum))

        gdl = (2.0 * weighted_intersection + self.smooth) / (
            weighted_denominator + self.smooth
        )
        return 1.0 - gdl


class WeightedCE_GDL_Loss(nn.Module):
    def __init__(
        self,
        class_weights=None,
        num_classes=3,
        ce_loss_weight=1.0,
        gdl_loss_weight=1.0,
        smooth=1e-6,
        ignore_index=-100,
    ):
        super().__init__()
        self.ce_loss_weight = float(ce_loss_weight)
        self.gdl_loss_weight = float(gdl_loss_weight)

        ce_weights = (
            torch.tensor(class_weights, dtype=torch.float32)
            if class_weights is not None
            else None
        )

        self.register_buffer("ce_weights", ce_weights, persistent=True)

        self.cross_entropy = nn.CrossEntropyLoss(
            weight=self.ce_weights, ignore_index=ignore_index
        )
        self.gdl = GeneralizedDiceLoss(
            num_classes=num_classes, smooth=smooth, ignore_index=ignore_index
        )

    def get_extra_state(self):
        return {
            "ce_loss_weight": self.ce_loss_weight,
            "gdl_loss_weight": self.gdl_loss_weight,
            "num_classes": self.gdl.num_classes,
            "smooth": self.gdl.smooth,
            "ignore_index": self.gdl.ignore_index,
        }

    def set_extra_state(self, state):
        if not state:
            return

        self.ce_loss_weight = float(state["ce_loss_weight"])
        self.gdl_loss_weight = float(state["gdl_loss_weight"])
        self.gdl.num_classes = int(state["num_classes"])
        self.gdl.smooth = float(state["smooth"])
        self.gdl.ignore_index = state["ignore_index"]
        self.cross_entropy.ignore_index = state["ignore_index"]

    def forward(self, logits, targets):
        ce_loss = self.cross_entropy(logits, targets)
        gdl_loss = self.gdl(logits, targets)
        return self.ce_loss_weight * ce_loss + self.gdl_loss_weight * gdl_loss

## Mean Teacher consistency loss

This notebook uses the authors' default **softmax-MSE consistency loss**:

- Paper: https://arxiv.org/abs/1703.01780
- Official loss implementation: https://github.com/CuriousAI/mean-teacher/blob/master/pytorch/mean_teacher/losses.py
- Official sigmoid ramp-up and EMA update: https://github.com/CuriousAI/mean-teacher

The original implementation was written for classification tensors `(B, C)`. Here, `reduction="mean"` applies the same normalization principle over segmentation tensors `(B, C, H, W)`. Gradients flow only through Student predictions; Teacher predictions are detached.


In [ ]:
class SoftmaxMSEConsistencyLoss(nn.Module):
    """
    Mean Teacher softmax-MSE consistency loss.

    Source:
        CuriousAI/mean-teacher, pytorch/mean_teacher/losses.py

    Segmentation adaptation:
        Mean reduction over B, C, H and W. This is equivalent to the
        authors' summed loss divided by batch size and spatial positions.
    """

    def forward(self, student_logits, teacher_logits):
        if student_logits.shape != teacher_logits.shape:
            raise ValueError(
                "Student and Teacher logits must have identical shapes, "
                f"got {student_logits.shape} and {teacher_logits.shape}."
            )

        student_probabilities = torch.softmax(student_logits, dim=1)
        teacher_probabilities = torch.softmax(
            teacher_logits.detach(),
            dim=1,
        )

        return F.mse_loss(
            student_probabilities,
            teacher_probabilities,
            reduction="mean",
        )

# Mean Teacher utils


In [ ]:
def create_teacher(student):
    """Create the EMA Teacher.

    Teacher stays in train mode during training so its BatchNorm layers use
    batch statistics just like the Student. Gradients remain disabled.
    """
    teacher = copy.deepcopy(student)
    teacher.requires_grad_(False)
    teacher.train()
    return teacher


@torch.no_grad()
def update_ema_teacher(student, teacher, ema_decay, global_step):
    """EMA-update Teacher parameters only (official Mean Teacher behavior).

    BatchNorm buffers are NOT copied from Student. While Teacher is in train
    mode, its own running statistics are updated by its weak-view forward pass.
    """
    effective_decay = min(
        float(ema_decay),
        1.0 - 1.0 / (float(global_step) + 1.0),
    )

    student_parameters = dict(student.named_parameters())
    teacher_parameters = dict(teacher.named_parameters())

    if student_parameters.keys() != teacher_parameters.keys():
        raise RuntimeError("Student and Teacher parameter structures do not match.")

    for name, teacher_parameter in teacher_parameters.items():
        teacher_parameter.mul_(effective_decay).add_(
            student_parameters[name],
            alpha=1.0 - effective_decay,
        )


def sigmoid_rampup(current, rampup_length):
    """Official Mean Teacher exponential sigmoid ramp-up."""
    rampup_length = float(rampup_length)
    if rampup_length == 0.0:
        return 1.0

    current = float(np.clip(current, 0.0, rampup_length))
    phase = 1.0 - current / rampup_length
    return float(np.exp(-5.0 * phase * phase))


def get_consistency_weight(epoch, mt_config):
    maximum = float(mt_config["max_consistency_weight"])
    rampup = int(mt_config["consistency_rampup_epochs"])
    return maximum * sigmoid_rampup(
        current=max(epoch - 1, 0),
        rampup_length=rampup,
    )

# Training and checkpoint utils


In [ ]:
def clone_state_dict_to_cpu(model):
    return {
        key: value.detach().cpu().clone() for key, value in model.state_dict().items()
    }


def component_type(component):
    if component is None:
        return None
    cls = component.__class__
    return f"{cls.__module__}.{cls.__qualname__}"


def empty_mean_teacher_history(num_classes=None):
    return {
        "supervised_loss": [],
        "consistency_loss": [],
        "weighted_consistency_loss": [],
        "consistency_to_supervised_ratio": [],
        "total_loss": [],
        "train_dice": [],
        "student_val_loss": [],
        "student_val_dice": [],
        "teacher_val_loss": [],
        "teacher_val_dice": [],
        "consistency_weight": [],
        "learning_rate": [],
    }


def _next_cycled_batch(loader, iterator):
    try:
        batch = next(iterator)
    except StopIteration:
        iterator = iter(loader)
        batch = next(iterator)
    return batch, iterator


def validate_resume_component(checkpoint, key, component):
    saved_type = checkpoint.get(f"{key}_type")
    current_type = component_type(component)
    if saved_type != current_type:
        raise ValueError(
            f"Cannot resume {key}: checkpoint uses {saved_type}, "
            f"but the notebook constructs {current_type}."
        )

In [ ]:
def save_mean_teacher_checkpoint(
    student,
    teacher,
    supervised_criterion,
    consistency_criterion,
    optimizer,
    scheduler,
    epoch,
    global_step,
    best_loss,
    best_dice,
    best_epoch,
    best_student_state_dict,
    best_teacher_state_dict,
    early_stopping_counter,
    history,
    config,
    mean_teacher_config,
    checkpoint_dir,
):
    checkpoint_dir = Path(checkpoint_dir)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    timestamp = datetime.now().strftime(config.get("timestamp_format", "%y%m%d_%H%M%S"))
    path = (
        checkpoint_dir
        / f"mean_teacher_mse_{student.__class__.__name__}_epoch_{epoch:03d}_{timestamp}.pth"
    )

    payload = {
        "training_method": "mean_teacher_softmax_mse",
        "epoch": int(epoch),
        "global_step": int(global_step),
        "student_type": component_type(student),
        "student_state_dict": student.state_dict(),
        "teacher_type": component_type(teacher),
        "teacher_state_dict": teacher.state_dict(),
        "supervised_criterion_type": component_type(supervised_criterion),
        "supervised_criterion_state_dict": supervised_criterion.state_dict(),
        "consistency_criterion_type": component_type(consistency_criterion),
        "consistency_criterion_state_dict": consistency_criterion.state_dict(),
        "optimizer_type": component_type(optimizer),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_type": component_type(scheduler),
        "scheduler_state_dict": (
            scheduler.state_dict() if scheduler is not None else None
        ),
        "best_loss": float(best_loss),
        "best_dice": float(best_dice),
        "best_epoch": int(best_epoch),
        "best_student_state_dict": best_student_state_dict,
        "best_teacher_state_dict": best_teacher_state_dict,
        "early_stopping_counter": int(early_stopping_counter),
        "history": history,
        "config": {
            key: value for key, value in config.items() if not str(key).startswith("_")
        },
        "mean_teacher_config": copy.deepcopy(mean_teacher_config),
    }

    torch.save(payload, path)
    print(f"[Checkpoint] Saved: {path}")
    return path


def load_mean_teacher_checkpoint(
    checkpoint_path,
    student,
    teacher,
    supervised_criterion,
    consistency_criterion,
    optimizer,
    scheduler,
    device,
):
    checkpoint = torch.load(
        checkpoint_path,
        map_location=device,
        weights_only=False,
    )

    if checkpoint.get("training_method") != "mean_teacher_softmax_mse":
        raise ValueError(
            "The selected checkpoint is not a softmax-MSE Mean Teacher checkpoint."
        )

    for key, component in (
        ("student", student),
        ("teacher", teacher),
        ("supervised_criterion", supervised_criterion),
        ("consistency_criterion", consistency_criterion),
        ("optimizer", optimizer),
    ):
        validate_resume_component(checkpoint, key, component)

    if scheduler is not None:
        validate_resume_component(checkpoint, "scheduler", scheduler)

    student.load_state_dict(checkpoint["student_state_dict"])
    teacher.load_state_dict(checkpoint["teacher_state_dict"])
    supervised_criterion.load_state_dict(checkpoint["supervised_criterion_state_dict"])
    consistency_criterion.load_state_dict(
        checkpoint["consistency_criterion_state_dict"]
    )
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

    if scheduler is not None and checkpoint.get("scheduler_state_dict") is not None:
        scheduler.load_state_dict(checkpoint["scheduler_state_dict"])

    teacher.requires_grad_(False)
    teacher.train()
    return checkpoint


def save_best_mean_teacher(
    student,
    teacher,
    best_student_state_dict,
    best_teacher_state_dict,
    best_loss,
    best_dice,
    best_epoch,
    history,
    config,
    mean_teacher_config,
):
    model_dir = Path(config["model_dir"])
    model_dir.mkdir(parents=True, exist_ok=True)
    path = (
        model_dir
        / f"best_mean_teacher_mse_{student.__class__.__name__}_epoch_{best_epoch}.pth"
    )

    torch.save(
        {
            "training_method": "mean_teacher_softmax_mse",
            "student_type": component_type(student),
            "student_state_dict": best_student_state_dict,
            "teacher_type": component_type(teacher),
            "teacher_state_dict": best_teacher_state_dict,
            "best_loss": float(best_loss),
            "best_dice": float(best_dice),
            "best_epoch": int(best_epoch),
            "history": history,
            "config": {
                key: value
                for key, value in config.items()
                if not str(key).startswith("_")
            },
            "mean_teacher_config": copy.deepcopy(mean_teacher_config),
            "class_names": CLASS_NAMES,
            "class_colors": CLASS_COLORS,
        },
        path,
    )
    print(f"[Best model] Saved: {path}")
    return path

## One epoch


In [ ]:
def train_mean_teacher_one_epoch(
    student,
    teacher,
    labeled_loader,
    unlabeled_loader,
    supervised_criterion,
    consistency_criterion,
    optimizer,
    device,
    epoch,
    global_step,
    num_classes,
    mean_teacher_config,
):
    """
    Consistency is computed on the combined (labeled + unlabeled) batch, not
    only on unlabeled data. This matches the original Mean Teacher design:
    the consistency cost does not need labels, so it is evaluated on every
    input, while the supervised cost only uses the labeled subset.

    Student and Teacher each still run a single forward pass per step
    (one over the concatenated batch) rather than one pass per subset, so
    BatchNorm statistics reflect the full batch composition.
    """
    student.train()
    teacher.train()

    consistency_weight = get_consistency_weight(epoch, mean_teacher_config)
    ema_decay = float(mean_teacher_config["ema_decay"])

    steps = max(len(labeled_loader), len(unlabeled_loader))
    labeled_iterator = iter(labeled_loader)
    unlabeled_iterator = iter(unlabeled_loader)

    supervised_sum = 0.0
    consistency_sum = 0.0
    weighted_consistency_sum = 0.0
    consistency_ratio_sum = 0.0
    total_sum = 0.0

    epoch_tp = np.zeros(num_classes, dtype=np.float64)
    epoch_fp = np.zeros(num_classes, dtype=np.float64)
    epoch_fn = np.zeros(num_classes, dtype=np.float64)

    for _ in range(steps):
        labeled_batch, labeled_iterator = _next_cycled_batch(
            labeled_loader,
            labeled_iterator,
        )
        unlabeled_batch, unlabeled_iterator = _next_cycled_batch(
            unlabeled_loader,
            unlabeled_iterator,
        )

        labeled_weak = labeled_batch["weak_image"].to(device, non_blocking=True)
        labeled_strong = labeled_batch["strong_image"].to(device, non_blocking=True)
        labeled_masks = labeled_batch["mask"].to(device, non_blocking=True).long()

        unlabeled_weak = unlabeled_batch["weak_image"].to(device, non_blocking=True)
        unlabeled_strong = unlabeled_batch["strong_image"].to(device, non_blocking=True)

        num_labeled = labeled_strong.size(0)

        student_input = torch.cat([labeled_strong, unlabeled_strong], dim=0)
        teacher_input = torch.cat([labeled_weak, unlabeled_weak], dim=0)

        optimizer.zero_grad(set_to_none=True)

        student_logits = student(student_input)

        with torch.no_grad():
            teacher_logits = teacher(teacher_input)

        supervised_loss = supervised_criterion(
            student_logits[:num_labeled],
            labeled_masks,
        )
        consistency_loss = consistency_criterion(
            student_logits,
            teacher_logits,
        )

        weighted_consistency_loss = consistency_weight * consistency_loss
        consistency_to_supervised_ratio = (
            weighted_consistency_loss.detach()
            / supervised_loss.detach().clamp_min(1e-8)
        )
        total_loss = supervised_loss + weighted_consistency_loss
        total_loss.backward()
        optimizer.step()

        global_step += 1
        update_ema_teacher(
            student,
            teacher,
            ema_decay=ema_decay,
            global_step=global_step,
        )

        with torch.no_grad():
            batch_tp, batch_fp, batch_fn = _confusion_counts_from_logits(
                student_logits[:num_labeled],
                labeled_masks,
                num_classes=num_classes,
            )
            epoch_tp += batch_tp
            epoch_fp += batch_fp
            epoch_fn += batch_fn

        supervised_sum += supervised_loss.item()
        consistency_sum += consistency_loss.item()
        weighted_consistency_sum += weighted_consistency_loss.item()
        consistency_ratio_sum += consistency_to_supervised_ratio.item()
        total_sum += total_loss.item()

    train_dice = _macro_dice_from_counts(
        epoch_tp,
        epoch_fp,
        epoch_fn,
        include_background=False,
    )

    metrics = {
        "supervised_loss": supervised_sum / max(steps, 1),
        "consistency_loss": consistency_sum / max(steps, 1),
        "weighted_consistency_loss": weighted_consistency_sum / max(steps, 1),
        "consistency_to_supervised_ratio": consistency_ratio_sum / max(steps, 1),
        "total_loss": total_sum / max(steps, 1),
        "train_dice": train_dice,
        "consistency_weight": consistency_weight,
    }
    return metrics, global_step


@torch.no_grad()
def validate_one_epoch(
    model,
    loader,
    criterion,
    device,
    num_classes=3,
):
    model.eval()
    running_loss = 0.0
    total_samples = 0

    epoch_tp = np.zeros(num_classes, dtype=np.float64)
    epoch_fp = np.zeros(num_classes, dtype=np.float64)
    epoch_fn = np.zeros(num_classes, dtype=np.float64)

    for batch in loader:
        images = batch["image"].to(device, non_blocking=True)
        masks = batch["mask"].to(device, non_blocking=True).long()

        logits = model(images)
        loss = criterion(logits, masks)
        batch_size = images.size(0)

        batch_tp, batch_fp, batch_fn = _confusion_counts_from_logits(
            logits,
            masks,
            num_classes=num_classes,
        )
        epoch_tp += batch_tp
        epoch_fp += batch_fp
        epoch_fn += batch_fn

        running_loss += loss.item() * batch_size
        total_samples += batch_size

    epoch_dice = _macro_dice_from_counts(
        epoch_tp,
        epoch_fp,
        epoch_fn,
        include_background=False,
    )

    return (
        running_loss / max(total_samples, 1),
        epoch_dice,
    )


## Training loop


In [ ]:
def train_mean_teacher(
    student,
    teacher,
    labeled_loader,
    unlabeled_loader,
    val_loader,
    supervised_criterion,
    consistency_criterion,
    optimizer,
    scheduler,
    config,
    mean_teacher_config,
    checkpoint_path=None,
):
    device = torch.device(config["device"])
    target_epoch = int(config["epochs"])
    num_classes = int(config["num_classes"])
    save_every = int(config.get("save_every", 0))

    early_stopping_enabled = bool(config.get("early_stopping", False))
    patience = int(config.get("early_stopping_patience", 10))
    min_delta = float(config.get("early_stopping_min_delta", 0.0))

    if patience < 1:
        raise ValueError("early_stopping_patience must be >= 1.")
    if min_delta < 0:
        raise ValueError("early_stopping_min_delta must be >= 0.")

    history = empty_mean_teacher_history(num_classes)
    start_epoch = 1
    global_step = 0
    best_loss = float("inf")
    best_dice = float("-inf")
    best_epoch = 0
    best_student_state_dict = None
    best_teacher_state_dict = None
    early_stopping_counter = 0

    if checkpoint_path is not None:
        checkpoint = load_mean_teacher_checkpoint(
            checkpoint_path=checkpoint_path,
            student=student,
            teacher=teacher,
            supervised_criterion=supervised_criterion,
            consistency_criterion=consistency_criterion,
            optimizer=optimizer,
            scheduler=scheduler,
            device=device,
        )

        start_epoch = int(checkpoint["epoch"]) + 1
        global_step = int(checkpoint.get("global_step", 0))
        best_loss = float(checkpoint.get("best_loss", float("inf")))
        best_epoch = int(checkpoint.get("best_epoch", 0))
        best_student_state_dict = checkpoint.get("best_student_state_dict")
        best_teacher_state_dict = checkpoint.get("best_teacher_state_dict")
        early_stopping_counter = int(checkpoint.get("early_stopping_counter", 0))
        history = checkpoint.get("history", history)

        saved_best_dice = checkpoint.get("best_dice")
        if saved_best_dice is not None:
            best_dice = float(saved_best_dice)
        elif best_epoch > 0 and len(history.get("teacher_val_dice", [])) >= best_epoch:
            # Backward-compatible resume from an older checkpoint whose
            # stored best state was selected by Teacher validation loss.
            best_dice = float(history["teacher_val_dice"][best_epoch - 1])

        expected_keys = empty_mean_teacher_history(num_classes).keys()
        completed_epochs = start_epoch - 1
        for key in expected_keys:
            values = history.setdefault(key, [])
            if len(values) < completed_epochs:
                values.extend([float("nan")] * (completed_epochs - len(values)))

        print(
            f"[Resume] epoch {start_epoch - 1} -> {target_epoch} | "
            f"global_step={global_step}"
        )
    else:
        print(f"[Train] epoch 1 -> {target_epoch}")

    if start_epoch > target_epoch:
        print(
            f"[Warning] No epoch to train: start={start_epoch}, "
            f"target={target_epoch}"
        )
        return student, teacher, history

    for epoch in range(start_epoch, target_epoch + 1):
        train_metrics, global_step = train_mean_teacher_one_epoch(
            student=student,
            teacher=teacher,
            labeled_loader=labeled_loader,
            unlabeled_loader=unlabeled_loader,
            supervised_criterion=supervised_criterion,
            consistency_criterion=consistency_criterion,
            optimizer=optimizer,
            device=device,
            epoch=epoch,
            global_step=global_step,
            num_classes=num_classes,
            mean_teacher_config=mean_teacher_config,
        )

        student_val_loss, student_val_dice = validate_one_epoch(
            student,
            val_loader,
            supervised_criterion,
            device,
            num_classes,
        )
        teacher_val_loss, teacher_val_dice = validate_one_epoch(
            teacher,
            val_loader,
            supervised_criterion,
            device,
            num_classes,
        )

        improved = teacher_val_dice > (best_dice + min_delta)
        if improved:
            best_loss = teacher_val_loss
            best_dice = teacher_val_dice
            best_epoch = epoch
            best_student_state_dict = clone_state_dict_to_cpu(student)
            best_teacher_state_dict = clone_state_dict_to_cpu(teacher)
            early_stopping_counter = 0
        elif early_stopping_enabled:
            early_stopping_counter += 1

        if scheduler is not None:
            if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step(teacher_val_loss)
            else:
                scheduler.step()

        current_lr = optimizer.param_groups[0]["lr"]

        for key in (
            "supervised_loss",
            "consistency_loss",
            "weighted_consistency_loss",
            "consistency_to_supervised_ratio",
            "total_loss",
            "train_dice",
            "consistency_weight",
        ):
            history[key].append(train_metrics[key])

        history["student_val_loss"].append(student_val_loss)
        history["student_val_dice"].append(student_val_dice)
        history["teacher_val_loss"].append(teacher_val_loss)
        history["teacher_val_dice"].append(teacher_val_dice)
        history["learning_rate"].append(current_lr)

        print(
            f"Epoch {epoch:02d}/{target_epoch:02d} | "
            f"sup={train_metrics['supervised_loss']:.5f} | "
            f"cons={train_metrics['consistency_loss']:.6f} | "
            f"w_cons={train_metrics['weighted_consistency_loss']:.6f} | "
            f"w_cons/sup={train_metrics['consistency_to_supervised_ratio']:.4f} | "
            f"lambda_c={train_metrics['consistency_weight']:.4f} | "
            f"student_val_dice={student_val_dice:.4f} | "
            f"teacher_val_dice={teacher_val_dice:.4f} | "
            f"lr={current_lr:.7f} | "
            f"best_teacher_val_dice={best_dice:.4f} @ {best_epoch}"
        )

        if save_every > 0 and epoch % save_every == 0:
            save_mean_teacher_checkpoint(
                student=student,
                teacher=teacher,
                supervised_criterion=supervised_criterion,
                consistency_criterion=consistency_criterion,
                optimizer=optimizer,
                scheduler=scheduler,
                epoch=epoch,
                global_step=global_step,
                best_loss=best_loss,
                best_dice=best_dice,
                best_epoch=best_epoch,
                best_student_state_dict=best_student_state_dict,
                best_teacher_state_dict=best_teacher_state_dict,
                early_stopping_counter=early_stopping_counter,
                history=history,
                config=config,
                mean_teacher_config=mean_teacher_config,
                checkpoint_dir=config["checkpoint_dir"],
            )

        if early_stopping_enabled and early_stopping_counter >= patience:
            print(
                f"[Early stopping] No Teacher validation-Dice improvement "
                f"for {patience} epochs. Best epoch: {best_epoch}."
            )
            break

    if best_teacher_state_dict is None or best_student_state_dict is None:
        best_student_state_dict = clone_state_dict_to_cpu(student)
        best_teacher_state_dict = clone_state_dict_to_cpu(teacher)
        best_epoch = epoch
        best_loss = teacher_val_loss
        best_dice = teacher_val_dice

    save_best_mean_teacher(
        student=student,
        teacher=teacher,
        best_student_state_dict=best_student_state_dict,
        best_teacher_state_dict=best_teacher_state_dict,
        best_loss=best_loss,
        best_dice=best_dice,
        best_epoch=best_epoch,
        history=history,
        config=config,
        mean_teacher_config=mean_teacher_config,
    )

    student.load_state_dict(best_student_state_dict)
    teacher.load_state_dict(best_teacher_state_dict)
    student.to(device)
    teacher.to(device).requires_grad_(False).eval()
    return student, teacher, history

# Visualization and evaluation


In [ ]:
def plot_mean_teacher_history(history):
    epochs = np.arange(1, len(history["total_loss"]) + 1)
    fig, axes = plt.subplots(2, 2, figsize=(16, 11))

    axes[0, 0].plot(epochs, history["supervised_loss"], label="Supervised")
    axes[0, 0].plot(epochs, history["consistency_loss"], label="Consistency (raw)")
    axes[0, 0].plot(
        epochs, history["weighted_consistency_loss"], label="Consistency (weighted)"
    )
    axes[0, 0].plot(epochs, history["total_loss"], label="Total")
    axes[0, 0].set_title("Training losses")
    axes[0, 0].legend()

    axes[0, 1].plot(epochs, history["train_dice"], label="Student train")
    axes[0, 1].plot(epochs, history["student_val_dice"], label="Student validation")
    axes[0, 1].plot(epochs, history["teacher_val_dice"], label="Teacher validation")
    axes[0, 1].set_title("Foreground Dice")
    axes[0, 1].legend()

    axes[1, 0].plot(
        epochs,
        history["student_val_loss"],
        label="Student validation",
    )
    axes[1, 0].plot(
        epochs,
        history["teacher_val_loss"],
        label="Teacher validation",
    )
    axes[1, 0].set_title("Validation supervised loss")
    axes[1, 0].legend()

    axes[1, 1].plot(
        epochs,
        history["consistency_weight"],
        label="Consistency weight",
    )
    axes[1, 1].plot(
        epochs,
        history["consistency_to_supervised_ratio"],
        label="Weighted consistency / supervised",
    )
    ax_lr = axes[1, 1].twinx()
    ax_lr.plot(
        epochs,
        history["learning_rate"],
        color="tab:orange",
        label="Learning rate",
    )
    axes[1, 1].set_title("Consistency weight and learning rate")
    axes[1, 1].set_ylabel("Consistency diagnostics")
    axes[1, 1].legend(loc="upper left")
    ax_lr.set_ylabel("Learning rate")

    for axis in axes.flat:
        axis.set_xlabel("Epoch")
        axis.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

In [ ]:
@torch.no_grad()
def evaluate_model(
    model,
    loader,
    device,
    num_classes=3,
    class_names=None,
):
    """
    Evaluate model on a dataset and compute metrics per class.

    Args:
        model: model to evaluate
        loader: DataLoader of the evaluation dataset
        device: device used for evaluation
        num_classes: number of segmentation classes
        class_names: dictionary mapping class IDs to class names

    Returns:
        DataFrame containing:
            - metrics for each class
            - macro_foreground_only:
                average of foreground classes only
            - macro_all_classes:
                average of all classes including background
    """

    class_names = class_names or {i: f"class_{i}" for i in range(num_classes)}

    tp = np.zeros(
        num_classes,
        dtype=np.float64,
    )

    fp = np.zeros(
        num_classes,
        dtype=np.float64,
    )

    fn = np.zeros(
        num_classes,
        dtype=np.float64,
    )

    hd95_values = {class_id: [] for class_id in range(num_classes)}

    model.eval()

    for batch in loader:
        images = batch["image"].to(
            device,
            non_blocking=True,
        )

        masks = (
            batch["mask"]
            .to(
                device,
                non_blocking=True,
            )
            .long()
        )

        logits = model(images)

        preds = torch.argmax(
            logits,
            dim=1,
        )

        preds_np = preds.cpu().numpy()
        masks_np = masks.cpu().numpy()

        for class_id in range(num_classes):
            pred_c = preds_np == class_id
            true_c = masks_np == class_id

            tp[class_id] += np.logical_and(
                pred_c,
                true_c,
            ).sum()

            fp[class_id] += np.logical_and(
                pred_c,
                ~true_c,
            ).sum()

            fn[class_id] += np.logical_and(
                ~pred_c,
                true_c,
            ).sum()

            for sample_idx in range(preds_np.shape[0]):
                hd95_values[class_id].append(
                    hd95_binary(
                        pred_c[sample_idx],
                        true_c[sample_idx],
                    )
                )

    rows = []

    for class_id in range(num_classes):
        dice = _safe_divide(
            2.0 * tp[class_id],
            (2.0 * tp[class_id] + fp[class_id] + fn[class_id]),
        )

        iou = _safe_divide(
            tp[class_id],
            (tp[class_id] + fp[class_id] + fn[class_id]),
        )

        precision = _safe_divide(
            tp[class_id],
            (tp[class_id] + fp[class_id]),
        )

        recall = _safe_divide(
            tp[class_id],
            (tp[class_id] + fn[class_id]),
        )

        hd_values = np.asarray(
            hd95_values[class_id],
            dtype=np.float64,
        )

        hd95 = float(hd_values.mean()) if hd_values.size > 0 else np.nan

        rows.append(
            {
                "class_id": class_id,
                "class_name": class_names.get(
                    class_id,
                    f"class_{class_id}",
                ),
                "dice": dice,
                "iou": iou,
                "hd95": hd95,
                "precision": precision,
                "recall": recall,
            }
        )

    result = pd.DataFrame(rows)

    # Macro foreground only
    foreground = result[result["class_id"].isin(range(1, num_classes))]

    macro_fg_row = {
        "class_id": "macro_fg",
        "class_name": "macro_foreground_only",
        "dice": foreground["dice"].mean(skipna=True),
        "iou": foreground["iou"].mean(skipna=True),
        "hd95": foreground["hd95"].mean(skipna=True),
        "precision": foreground["precision"].mean(skipna=True),
        "recall": foreground["recall"].mean(skipna=True),
    }

    # Macro all classes includes background
    all_classes = result[result["class_id"].isin(range(num_classes))]

    macro_all_row = {
        "class_id": "macro_all",
        "class_name": "macro_all_classes",
        "dice": all_classes["dice"].mean(skipna=True),
        "iou": all_classes["iou"].mean(skipna=True),
        "hd95": all_classes["hd95"].mean(skipna=True),
        "precision": all_classes["precision"].mean(skipna=True),
        "recall": all_classes["recall"].mean(skipna=True),
    }

    return pd.concat(
        [
            result,
            pd.DataFrame(
                [
                    macro_fg_row,
                    macro_all_row,
                ]
            ),
        ],
        ignore_index=True,
    )

# Main


## Setup


In [ ]:
if RESUME_CHECKPOINT is None:
    CONFIG = copy.deepcopy(DEFAULT_CONFIG)
    MT_CONFIG = copy.deepcopy(DEFAULT_MEAN_TEACHER_CONFIG)
else:
    resume_path = Path(RESUME_CHECKPOINT)
    if not resume_path.exists():
        raise FileNotFoundError(f"Resume checkpoint not found: {resume_path}")

    checkpoint_metadata = torch.load(
        resume_path,
        map_location="cpu",
        weights_only=False,
    )

    if checkpoint_metadata.get("training_method") != "mean_teacher_softmax_mse":
        raise ValueError(
            "The selected checkpoint is not a softmax-MSE Mean Teacher checkpoint."
        )

    checkpoint_config = checkpoint_metadata.get("config")
    checkpoint_mt_config = checkpoint_metadata.get("mean_teacher_config")

    if not isinstance(checkpoint_config, dict):
        raise KeyError("Checkpoint does not contain a valid 'config' dictionary.")
    if not isinstance(checkpoint_mt_config, dict):
        raise KeyError(
            "Checkpoint does not contain a valid 'mean_teacher_config' dictionary."
        )

    CONFIG = copy.deepcopy(checkpoint_config)
    MT_CONFIG = copy.deepcopy(checkpoint_mt_config)

CONFIG["epochs"] = int(DEFAULT_CONFIG["epochs"])
CONFIG["checkpoint_dir"] = str(CHECKPOINT_DIR)
CONFIG["model_dir"] = str(MODEL_DIR)

print("Config source              :", "Checkpoint" if RESUME_CHECKPOINT else "Default")
print("Device                     :", CONFIG["device"])
print("Train subset               :", CONFIG["train_subset"])
print("Target epochs              :", CONFIG["epochs"])
print("EMA decay                  :", MT_CONFIG["ema_decay"])
print("Max consistency weight     :", MT_CONFIG["max_consistency_weight"])
print("Consistency ramp-up epochs :", MT_CONFIG["consistency_rampup_epochs"])

## Download


In [ ]:
dataset_path = kagglehub.dataset_download("avcuongy/bkai-igh-neopolyp-small-sample")
DATA_DIR = Path(dataset_path) / "data"
SPLIT_DIR = DATA_DIR / "sample"

TRAIN_CSV_PATH = SPLIT_DIR / "train.csv"
VALID_CSV_PATH = SPLIT_DIR / "valid.csv"
TEST_CSV_PATH = SPLIT_DIR / "test.csv"

train_df = pd.read_csv(TRAIN_CSV_PATH)
valid_df = pd.read_csv(VALID_CSV_PATH)
test_df = pd.read_csv(TEST_CSV_PATH)

print("Data directory:", DATA_DIR)
print("Train:", train_df.shape)
print("Valid:", valid_df.shape)
print("Test :", test_df.shape)

In [ ]:
TRAIN_SUBSET_FLAG_MAP = {
    "half": "labeled_1_2",
    "quarter": "labeled_1_4",
}

train_subset = str(CONFIG["train_subset"]).strip().lower()
if train_subset not in TRAIN_SUBSET_FLAG_MAP:
    raise ValueError(
        "Mean Teacher requires an unlabeled pool. "
        "CONFIG['train_subset'] must be 'half' or 'quarter'."
    )

flag_column = TRAIN_SUBSET_FLAG_MAP[train_subset]
if flag_column not in train_df.columns:
    raise KeyError(f"Missing split column: {flag_column}")

flag_values = pd.to_numeric(train_df[flag_column], errors="coerce")
if flag_values.isna().any():
    raise ValueError(f"Column '{flag_column}' contains missing/non-numeric values.")

unique_flags = set(flag_values.astype(int).unique().tolist())
if not unique_flags.issubset({0, 1}):
    raise ValueError(
        f"Column '{flag_column}' must contain only 0/1. "
        f"Found: {sorted(unique_flags)}"
    )

labeled_train_df = train_df[flag_values.eq(1)].copy().reset_index(drop=True)
unlabeled_train_df = train_df[flag_values.eq(0)].copy().reset_index(drop=True)

if labeled_train_df.empty:
    raise ValueError("The labeled training split is empty.")
if unlabeled_train_df.empty:
    raise ValueError("The unlabeled training split is empty.")

print("Split column       :", flag_column)
print("Labeled training   :", len(labeled_train_df))
print("Unlabeled training :", len(unlabeled_train_df))
print("Validation         :", len(valid_df))
print("Test               :", len(test_df))

## DataLoaders


In [ ]:
augmentation_seed = int(CONFIG["seed"])

labeled_shared_geometry = build_labeled_shared_geometry(
    CONFIG["image_height"],
    CONFIG["image_width"],
    seed=augmentation_seed + 0,
)
labeled_weak_transform = build_labeled_weak_transform(seed=augmentation_seed + 1)
labeled_strong_transform = build_labeled_strong_transform(seed=augmentation_seed + 2)
unlabeled_shared_geometry = build_unlabeled_shared_geometry(
    CONFIG["image_height"],
    CONFIG["image_width"],
    seed=augmentation_seed + 3,
)
unlabeled_weak_transform = build_unlabeled_weak_transform(seed=augmentation_seed + 4)
unlabeled_strong_transform = build_unlabeled_strong_transform(
    seed=augmentation_seed + 5
)
eval_transform = build_eval_transform(
    CONFIG["image_height"],
    CONFIG["image_width"],
    seed=augmentation_seed + 6,
)

labeled_dataset = MeanTeacherLabeledDataset(
    dataframe=labeled_train_df,
    image_dir=DATA_DIR,
    mask_dir=DATA_DIR,
    shared_geometry=labeled_shared_geometry,
    weak_transform=labeled_weak_transform,
    strong_transform=labeled_strong_transform,
    image_col=CONFIG["image_col"],
    mask_col=CONFIG["mask_col"],
)
unlabeled_dataset = MeanTeacherUnlabeledDataset(
    dataframe=unlabeled_train_df,
    image_dir=DATA_DIR,
    shared_geometry=unlabeled_shared_geometry,
    weak_transform=unlabeled_weak_transform,
    strong_transform=unlabeled_strong_transform,
    image_col=CONFIG["image_col"],
)
valid_dataset = PolypSegmentationDataset(
    dataframe=valid_df,
    image_dir=DATA_DIR,
    mask_dir=DATA_DIR,
    transform=eval_transform,
    image_col=CONFIG["image_col"],
    mask_col=CONFIG["mask_col"],
    has_mask=True,
)
test_dataset = PolypSegmentationDataset(
    dataframe=test_df,
    image_dir=DATA_DIR,
    mask_dir=DATA_DIR,
    transform=eval_transform,
    image_col=CONFIG["image_col"],
    mask_col=CONFIG["mask_col"],
    has_mask=True,
)

# Fail before training if CSV columns or representative mask paths are wrong.
validate_labeled_dataset(labeled_dataset, "labeled_train", max_samples=8)
validate_labeled_dataset(valid_dataset, "validation", max_samples=8)
validate_labeled_dataset(test_dataset, "test", max_samples=8)

In [ ]:
labeled_generator = torch.Generator()
labeled_generator.manual_seed(CONFIG["seed"])

unlabeled_generator = torch.Generator()
unlabeled_generator.manual_seed(CONFIG["seed"] + 1)

common_loader_args = {
    "num_workers": CONFIG["num_workers"],
    "pin_memory": torch.cuda.is_available(),
    "worker_init_fn": seed_worker,
}

labeled_loader = DataLoader(
    labeled_dataset,
    batch_size=CONFIG["labeled_batch_size"],
    shuffle=True,
    generator=labeled_generator,
    **common_loader_args,
)
unlabeled_loader = DataLoader(
    unlabeled_dataset,
    batch_size=CONFIG["unlabeled_batch_size"],
    shuffle=True,
    generator=unlabeled_generator,
    **common_loader_args,
)
valid_loader = DataLoader(
    valid_dataset,
    batch_size=CONFIG["eval_batch_size"],
    shuffle=False,
    **common_loader_args,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=CONFIG["eval_batch_size"],
    shuffle=False,
    **common_loader_args,
)

print("Labeled batches  :", len(labeled_loader))
print("Unlabeled batches:", len(unlabeled_loader))
print("Validation batches:", len(valid_loader))
print("Test batches      :", len(test_loader))

## Model


In [ ]:
device = torch.device(CONFIG["device"])

student = UNet(
    backbone=CONFIG["backbone"],
    n_classes=CONFIG["num_classes"],
    pretrained=CONFIG["pretrained"],
    bilinear=CONFIG["bilinear"],
).to(device)

if CONFIG.get("freeze_encoder", False):
    student.freeze_encoder()
else:
    student.unfreeze_encoder()

teacher = create_teacher(student).to(device)

In [ ]:
supervised_criterion = WeightedCE_GDL_Loss(
    class_weights=[0.2, 2.0, 1.0],
    num_classes=CONFIG["num_classes"],
    ce_loss_weight=1.0,
    gdl_loss_weight=2.0,
    smooth=1e-8,
    ignore_index=-100,
).to(device)

consistency_criterion = SoftmaxMSEConsistencyLoss().to(device)

optimizer = torch.optim.AdamW(
    student.parameters(),
    lr=3e-4,
    weight_decay=1e-4,
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=CONFIG["epochs"],
    eta_min=1e-6,
)

In [ ]:
print("Student                 :", student.__class__.__name__)
print("Teacher                 :", teacher.__class__.__name__)
print(
    "Teacher trainable params:",
    sum(p.numel() for p in teacher.parameters() if p.requires_grad),
)
print("Supervised criterion    :", supervised_criterion.__class__.__name__)
print("Consistency criterion   :", consistency_criterion.__class__.__name__)
print("Optimizer               :", optimizer.__class__.__name__)
print("Scheduler               :", scheduler.__class__.__name__)
print("Mean Teacher settings   :", MT_CONFIG)

## Train


In [ ]:
student, teacher, history = train_mean_teacher(
    student=student,
    teacher=teacher,
    labeled_loader=labeled_loader,
    unlabeled_loader=unlabeled_loader,
    val_loader=valid_loader,
    supervised_criterion=supervised_criterion,
    consistency_criterion=consistency_criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    config=CONFIG,
    mean_teacher_config=MT_CONFIG,
    checkpoint_path=RESUME_CHECKPOINT,
)

## Plot loss


In [ ]:
plot_mean_teacher_history(history)

## Eval


In [ ]:
student_evaluation = evaluate_model(
    model=student,
    loader=test_loader,
    device=device,
    num_classes=CONFIG["num_classes"],
    class_names=CLASS_NAMES,
)

teacher_evaluation = evaluate_model(
    model=teacher,
    loader=test_loader,
    device=device,
    num_classes=CONFIG["num_classes"],
    class_names=CLASS_NAMES,
)

In [ ]:
print("Student test metrics")
display(student_evaluation.round(4))

In [ ]:
print("Teacher test metrics")
display(teacher_evaluation.round(4))